# 2D Resonance Imaging with PLEIADES

This notebook demonstrates the 2D imaging pipeline for spatially-resolved
neutron resonance analysis. The pipeline fits each pixel of a hyperspectral
image independently using SAMMY, then aggregates the results into isotope
abundance maps.

## Pipeline Overview

```
TIFF data  →  HyperspectralLoader  →  pixel spectra
                                           ↓
                              BatchFittingOrchestrator (parallel SAMMY)
                                           ↓
                                   ResultsAggregator  →  2D maps
                                           ↓
                          AbundanceMapGenerator + AbundanceMapVisualizer
```

**Two usage levels:**
1. **High-level**: `analyze_imaging()` — single function call for the entire pipeline
2. **Component-level**: Step-by-step for customization and inspection

This notebook walks through both approaches using the test TIFF stack and
synthetic fit results (no SAMMY executable required).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from pleiades.imaging import (
    HyperspectralLoader,
    ResultsAggregator,
    AbundanceMapGenerator,
    AbundanceMapVisualizer,
    Imaging2DResults,
    PixelFitResult,
)
from pleiades.imaging.config import ImagingConfig

## 1. Loading Hyperspectral Data

The `HyperspectralLoader` reads multi-page TIFF files where each page is one
energy bin and spatial dimensions form the image. The test file is a
500×256×256 transmission stack.

In [ ]:
tiff_path = Path("../../tests/data/pleiades_data/LANL-ORNL_example.tif")
print(f"TIFF file exists: {tiff_path.exists()}")

# Energy axis (placeholder — real analysis would use calibrated values)
energy = np.linspace(1.0, 200.0, 500)

loader = HyperspectralLoader(tiff_path, energy=energy)
hyperspectral = loader.load()

n_energy, height, width = hyperspectral.shape
print(f"Image shape: {height}×{width} pixels, {n_energy} energy bins")
print(f"Total pixels: {hyperspectral.n_pixels:,}")
print(f"Energy range: {energy[0]:.1f} – {energy[-1]:.1f} eV")

### Visualize the raw data

Look at the spatial image at a single energy bin and a single pixel's
transmission spectrum.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Spatial image at energy bin 140 (strong resonance)
bin_idx = 140
im = axes[0].imshow(hyperspectral.data[bin_idx], cmap="viridis", origin="upper")
axes[0].set_title(f"Transmission at energy bin {bin_idx} ({energy[bin_idx]:.1f} eV)")
fig.colorbar(im, ax=axes[0], label="Transmission")

# Spectrum for center pixel
center_row, center_col = height // 2, width // 2
spectrum = hyperspectral.data[:, center_row, center_col]
axes[1].plot(energy, spectrum, linewidth=0.5)
axes[1].set_xlabel("Energy (eV)")
axes[1].set_ylabel("Transmission")
axes[1].set_title(f"Pixel ({center_row}, {center_col}) spectrum")
axes[1].set_xscale("log")

fig.tight_layout()
plt.show()

### Iterate pixels

The loader provides `iter_pixels()` to yield `PixelSpectrum` objects.
Use `roi=(x1, y1, x2, y2)` to restrict to a sub-region.

In [ ]:
# Extract a small 4×4 ROI for demonstration
roi = (120, 120, 124, 124)  # (x1, y1, x2, y2)
pixels = list(loader.iter_pixels(roi=roi))

print(f"ROI pixels: {len(pixels)}")
print(f"First pixel: row={pixels[0].row}, col={pixels[0].col}")
print(f"  Energy shape: {pixels[0].energy.shape}")
print(f"  Transmission range: [{pixels[0].transmission.min():.3f}, {pixels[0].transmission.max():.3f}]")

## 2. Configuration

`ImagingConfig` holds the isotope list, material properties, and energy range
needed by the SAMMY fitting engine.

In [ ]:
config = ImagingConfig(
    isotopes=["Ta-181"],
    element="Ta",
    mass_number=181,
    density_g_cm3=16.65,
    thickness_mm=0.127,
    atomic_mass_amu=180.94788,
    natural_abundances=True,
    min_energy_eV=1.0,
    max_energy_eV=200.0,
    temperature_K=293.6,
)

print(f"Isotopes: {config.isotopes}")
print(f"Material: {config.element}-{config.mass_number}")
print(f"  Density: {config.density_g_cm3} g/cm\u00b3")
print(f"  Thickness: {config.thickness_mm} mm")
print(f"Energy range: {config.min_energy_eV}–{config.max_energy_eV} eV")

## 3. Batch Fitting (Simulated)

In production, `BatchFittingOrchestrator` runs SAMMY for every pixel in
parallel. Here we simulate the results with synthetic abundance values
to demonstrate the downstream pipeline.

We create a ground-truth abundance map with a spatial gradient, then build
`PixelFitResult` objects as if SAMMY had produced them.

In [ ]:
from pleiades.nuclear.isotopes.models import IsotopeInfo, IsotopeMassData
from pleiades.nuclear.models import IsotopeParameters, nuclearParameters
from pleiades.sammy.results.models import FitResults


def make_pixel_result(row, col, abundance, chi_sq=1.5):
    """Create a synthetic PixelFitResult for one pixel."""
    isotope_info = IsotopeInfo(
        name="Ta-181",
        atomic_number=73,
        mass_number=181,
        mass_data=IsotopeMassData(atomic_mass=180.948),
        abundance=abundance * 100.0,  # stored as percent
        spin=3.5,
    )
    isotope_params = IsotopeParameters(
        isotope_information=isotope_info,
        abundance=abundance,
    )
    nuclear_params = nuclearParameters(isotopes=[isotope_params])
    fit = FitResults(nuclear_data=nuclear_params)

    return PixelFitResult(
        row=row, col=col,
        fit_results=fit,
        success=True,
        chi_squared=chi_sq,
    )

In [ ]:
# Create ground-truth: horizontal gradient from 0.2 to 0.9 across the full image
ta_truth = np.linspace(0.2, 0.9, width)[np.newaxis, :].repeat(height, axis=0)

# Add some Gaussian noise to simulate fit uncertainty
rng = np.random.default_rng(42)
chi_sq_map = rng.uniform(1.0, 3.0, (height, width))

# Build PixelFitResult for every pixel
all_pixels = list(loader.iter_pixels())
pixel_results = [
    make_pixel_result(
        p.row, p.col,
        abundance=ta_truth[p.row, p.col],
        chi_sq=chi_sq_map[p.row, p.col],
    )
    for p in all_pixels
]

print(f"Generated {len(pixel_results):,} synthetic fit results")
print(f"Ground truth abundance range: [{ta_truth.min():.2f}, {ta_truth.max():.2f}]")

## 4. Results Aggregation

The `ResultsAggregator` takes the list of per-pixel results and builds
2D abundance maps, a chi-squared map, and a success mask.

In [ ]:
aggregator = ResultsAggregator(
    isotope_names=["Ta-181"],
    height=height,
    width=width,
)
results = aggregator.aggregate(pixel_results, hyperspectral)

print(f"Abundance maps shape: {results.abundance_maps.shape}")
print(f"Chi-squared map shape: {results.chi_squared_map.shape}")
print(f"Success rate: {results.success_mask.sum() / results.success_mask.size:.1%}")
print(f"Isotopes: {results.isotope_names}")

## 5. Abundance Map Generation and Visualization

`AbundanceMapGenerator` extracts individual isotope maps from the aggregated
results. `AbundanceMapVisualizer` provides publication-quality plots.

In [ ]:
gen = AbundanceMapGenerator(results)
ta_map = gen.generate_map("Ta-181")

print(f"Ta-181 map shape: {ta_map.shape}")
print(f"Abundance range: [{np.nanmin(ta_map):.4f}, {np.nanmax(ta_map):.4f}]")

# Verify recovery of ground truth
max_error = np.nanmax(np.abs(ta_map - ta_truth))
print(f"Max deviation from ground truth: {max_error:.2e}")

In [ ]:
viz = AbundanceMapVisualizer(results)

fig, ax = viz.plot_single_isotope("Ta-181", cmap="plasma")
ax.set_xlabel("Column")
ax.set_ylabel("Row")
plt.show()

### Quality overlay

Overlay the chi-squared map on the abundance map to identify regions with
poor fit quality.

In [ ]:
fig, ax = viz.plot_quality_overlay("Ta-181", quality_metric="chi_squared", alpha=0.4)
ax.set_xlabel("Column")
ax.set_ylabel("Row")
plt.show()

## 6. Saving and Loading Results

Results can be saved to HDF5 and reloaded later for visualization or
further analysis.

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as tmpdir:
    h5_path = Path(tmpdir) / "imaging_results.h5"
    results.save_hdf5(h5_path)
    print(f"Saved to {h5_path} ({h5_path.stat().st_size / 1024:.0f} KB)")

    # Reload
    loaded = Imaging2DResults.load_hdf5(h5_path, source_hyperspectral=hyperspectral)
    print(f"Loaded isotopes: {loaded.isotope_names}")
    print(f"Maps shape: {loaded.abundance_maps.shape}")

    # Verify round-trip fidelity
    np.testing.assert_array_equal(results.abundance_maps, loaded.abundance_maps)
    print("Round-trip verification: PASSED")

## 7. High-Level API: `analyze_imaging()`

For production use with a SAMMY executable installed, the entire pipeline
collapses to a single function call:

```python
from pleiades.imaging import analyze_imaging
from pleiades.imaging.config import ImagingConfig

config = ImagingConfig(
    isotopes=["Ta-181"],
    element="Ta",
    mass_number=181,
    density_g_cm3=16.65,
    thickness_mm=0.127,
    atomic_mass_amu=180.94788,
    min_energy_eV=1.0,
    max_energy_eV=200.0,
)

results = analyze_imaging(
    source="/path/to/hyperspectral.tif",
    imaging_config=config,
    sammy_executable=Path("/path/to/sammy"),
    energy=energy_array,
    n_workers=8,
    roi=(100, 100, 200, 200),         # optional sub-region
    checkpoint_file=Path("ckpt.pkl"),  # resume on interrupt
    save_path=Path("results.h5"),      # auto-save
)

# Visualize
viz = AbundanceMapVisualizer(results)
fig, ax = viz.plot_single_isotope("Ta-181")
```

### Key parameters

| Parameter | Description | Default |
|-----------|-------------|---------|
| `source` | Path to TIFF file or directory | *required* |
| `imaging_config` | Isotopes and material properties | *required* |
| `sammy_executable` | Path to SAMMY binary | *required* |
| `energy` | Energy axis (eV); inferred if `None` | `None` |
| `n_workers` | Parallel SAMMY processes | `4` |
| `roi` | Sub-region `(x1, y1, x2, y2)` | `None` (all) |
| `checkpoint_file` | Save/resume checkpoint | `None` |
| `resume` | Resume from existing checkpoint | `False` |
| `timeout_per_job` | Max seconds per pixel | `None` |
| `max_retries` | Retry failed pixels | `0` |
| `save_path` | Auto-save results to HDF5 | `None` |

## 8. Multi-Isotope Example

For samples containing multiple isotopes, the pipeline produces one
abundance map per isotope. Here we demonstrate with synthetic two-isotope
data.

In [ ]:
# Two-isotope ground truth: Ta-181 gradient + W-182 inverse gradient
ta_truth_2 = np.linspace(0.3, 0.7, width)[np.newaxis, :].repeat(height, axis=0)
w_truth_2 = 1.0 - ta_truth_2  # complementary abundance


def make_two_isotope_result(row, col, ta_abund, w_abund, chi_sq=2.0):
    """Create a PixelFitResult with Ta-181 and W-182."""
    ta_info = IsotopeInfo(
        name="Ta-181",
        atomic_number=73,
        mass_number=181,
        mass_data=IsotopeMassData(atomic_mass=180.948),
        abundance=ta_abund * 100.0,
        spin=3.5,
    )
    w_info = IsotopeInfo(
        name="W-182",
        atomic_number=74,
        mass_number=182,
        mass_data=IsotopeMassData(atomic_mass=181.948),
        abundance=w_abund * 100.0,
        spin=0.0,
    )
    nuclear = nuclearParameters(
        isotopes=[
            IsotopeParameters(isotope_information=ta_info, abundance=ta_abund),
            IsotopeParameters(isotope_information=w_info, abundance=w_abund),
        ]
    )
    fit = FitResults(nuclear_data=nuclear)
    return PixelFitResult(row=row, col=col, fit_results=fit, success=True, chi_squared=chi_sq)


multi_results = [
    make_two_isotope_result(
        p.row, p.col,
        ta_abund=ta_truth_2[p.row, p.col],
        w_abund=w_truth_2[p.row, p.col],
    )
    for p in all_pixels
]

agg_multi = ResultsAggregator(isotope_names=["Ta-181", "W-182"], height=height, width=width)
results_multi = agg_multi.aggregate(multi_results, hyperspectral)

print(f"Isotopes: {results_multi.isotope_names}")
print(f"Abundance maps shape: {results_multi.abundance_maps.shape}")

In [ ]:
viz_multi = AbundanceMapVisualizer(results_multi)
fig, axes = viz_multi.plot_multi_isotope(ncols=2, figsize=(10, 4))
plt.show()

## Summary

The PLEIADES 2D imaging pipeline provides:

- **`HyperspectralLoader`** — Load TIFF stacks, NeXus files, or directories of frames
- **`ImagingConfig`** — Isotope and material property configuration
- **`BatchFittingOrchestrator`** — Parallel SAMMY fitting with checkpointing
- **`ResultsAggregator`** — Build 2D abundance/quality maps from pixel results
- **`AbundanceMapGenerator`** — Extract individual isotope maps
- **`AbundanceMapVisualizer`** — Publication-quality abundance and quality-overlay plots
- **`analyze_imaging()`** — One-call convenience function for the full pipeline
- **HDF5 persistence** — Save/load results for later analysis